In [38]:
import numpy as np
import time


In [252]:
dx = 0.08
N = 1/dx + 1
# print(N)
N_int = int(N-2)
print(N_int)
L = int(N_int**2)
print(L)
A = np.zeros((L,L))
B = np.zeros((L,1))

11
121


In [253]:
boundary = {"top": 1, "bottom": 0, "left": 0, "right": 0}

In [254]:
for i in range(N_int):
    for j in range(N_int):
        ipt = i*(N_int) + j

        A[ipt,ipt] = -4

        if i>0:
            A[ipt,ipt - N_int] = 1
        else:
            B[ipt] -= boundary["top"]
        
        if i<N_int-1:
            A[ipt,ipt + N_int] = 1
        else:
            B[ipt] -= boundary["bottom"]

        if j>0:
            A[ipt,ipt - 1] = 1
        else:
            B[ipt] -= boundary["left"]
        
        if j<N_int-1:
            A[ipt,ipt + 1] = 1
        else:  
            B[ipt] -= boundary["right"]





In [255]:
print(A.shape,B.shape)

(121, 121) (121, 1)


In [256]:
def SOR(A, B, omega=1.25, max_iterations=1000, tolerance=1e-6):
    start = time.perf_counter()
    D = np.diag(np.diag(A))
    # F = -(np.triu(A) - D)
    E = -(np.tril(A) - D)
    M = (1/omega)*D - E
    G = np.linalg.inv(M) @ (M - A)
    f = np.linalg.inv(M) @ B
    f = np.array([f]).T

    # print("Spectral Radius (SOR):", max(abs(np.linalg.eigvals(G))))
    # print("Convergence rate (SOR):", -np.log10(max(abs(np.linalg.eigvals(G)))))
    x = np.zeros((B.shape[0], 1))  # Reset initial guess for each omega

    for iteration in range(max_iterations):
        
        x_new = G @ x + f
        if np.linalg.norm(B-A@x) < tolerance:
            print(f"SOR method converged in {iteration} iterations.")
            break
        # print("Iteration:", iteration,"Difference:", np.linalg.norm(x_new - x))
        x = x_new

    print("CPU Time(SOR):", round(time.perf_counter() - start, 6), "seconds")
    return x
    # ange(max_iterations):


In [257]:
def steepest_desc(A,b,tol=1e-6,maxiter=1000):
    start = time.perf_counter()
    x = np.zeros((len(b),1))
    r = b - A @ x
    p = A @ r
    for epoch in range(maxiter):
        # print(epoch, np.linalg.norm(r))
        if np.linalg.norm(r) < tol:
            print("Steepest Descent converged in", epoch, "iterations")
            break
        alpha = (r.T @ r) / (p.T @ r)
        x = x + alpha * r
        r = b - A @ x
        p = A @ r
    print("CPU compute time (steepest_desc):", round(time.perf_counter() - start, 6), "seconds")
    return x


In [258]:
def min_res(A,b,tol=1e-6,maxiter=1000):
    start = time.perf_counter()
    x = np.zeros((len(b),1))
    r = b - A@x
    p = A @ r
    for epoch in range(maxiter):
        if np.linalg.norm(r) < tol:
            print("Min Residual converged in", epoch, "iterations")
            break
        alpha = (p.T@r)/(p.T@p)
        x = x + alpha*r
        r = r - alpha*p
        p = A @ r
    print("CPU compute time (min_res):", round(time.perf_counter() - start, 6), "seconds")
    return x

In [259]:
def conj_grad(A,b,tol=1e-6,maxiter=1000):
    start = time.perf_counter()
    x = np.zeros((len(b),1))
    r = b - A@x
    p = r
    for epoch in range(maxiter):
        if np.linalg.norm(r) < tol:
            print("Conjugate Gradient converged in", epoch, "iterations")
            break
        Ap = A @ p
        alpha = (r.T@r)/(p.T@Ap)
        x = x + alpha*p
        r_new = r - alpha*Ap
        beta = (r_new.T@r_new)/(r.T@r)
        p = r_new + beta*p
        r = r_new
    print("CPU compute time (conj_grad):", round(time.perf_counter() - start, 6), "seconds")
    return x


In [260]:
def bicgstab(A,b,tol=1e-6,maxiter=1000):
    start = time.perf_counter()
    x = np.zeros((len(b),1))
    r = b - A @ x
    r_star  = np.random.rand(len(b))
    # r_star = r.copy()
    p = r.copy()
    
    for epoch in range(maxiter):
        # print(epoch, np.linalg.norm(r))
        if np.linalg.norm(r) < tol:
            print("BICGSTAB converged in", epoch, "iterations")
            break
        Ap = A @ p
        alpha = (r.T @r_star)/(r_star.T @Ap)
        s = r - alpha*Ap
        As = A @ s
        w = (s.T @ As) / (As.T @ As)
        x = x + alpha*p + w*s

        r_new = s - w*As
        beta = ((r_new.T @ r_star)/(r.T @ r_star))*(alpha/w)

        p = r_new + beta*(p - w*Ap)
        r = r_new
    print("CPU Time(BICGSTAB):", round(time.perf_counter() - start,6), "seconds")
    return x

In [261]:
D = np.diag(np.diag(A))
M = D 
G = np.linalg.inv(M) @ (M - A)
spec_rad = max(abs(np.linalg.eigvals(G)))
w_opt = 2/(1 + np.sqrt(1 - spec_rad**2))
print("Optimal omega:", w_opt)

Optimal omega: 1.5887907064808888


In [262]:
x_sor = SOR(A,B,w_opt,max_iterations=10000)
x_sd = steepest_desc(A,B,maxiter=10000)
x_mr = min_res(A,B,maxiter=10000)
x_cg = conj_grad(A,B,maxiter=10000)
x_bicg = bicgstab(A,B,maxiter=10000)

SOR method converged in 31 iterations.
CPU Time(SOR): 0.004717 seconds
Steepest Descent converged in 374 iterations
CPU compute time (steepest_desc): 0.011944 seconds
Min Residual converged in 365 iterations
CPU compute time (min_res): 0.010275 seconds
Conjugate Gradient converged in 28 iterations
CPU compute time (conj_grad): 0.001099 seconds
BICGSTAB converged in 20 iterations
CPU Time(BICGSTAB): 0.001393 seconds
